# Task 4: Game title detection (5 classes, macro F1): ensemble and class-prior step

**Goal.** Identify which of 5 games a screenshot shows. Metric: macro F1. Pretrained models are allowed for this task.

**What this notebook does.** The final prediction is a probability-average ensemble of 25 fine-tuned backbones followed by a class-prior (Saerens EM) correction:
1. Each member was trained by `train_backbone.py` (about 1 to 8 GPU hours each; the exact commands are in `train_all_members.sh`) and saved its test probabilities (`t4_<tag>_test_prob.npy`).
2. Members are averaged with integer weights: six DINOv2-L trained with the new recipe (33 each), one DINOv2-B new recipe (20), ten DINOv2-L baseline seeds (7 each), six ConvNeXt-B/DINOv2-B seeds (4 each), ConvNeXt-S/T (2 each). The new-recipe members carry 69% of the weight.
3. **Class-prior correction.** The test class shares are estimated from the ensemble probabilities with the Saerens EM algorithm (no labels), and the probabilities are multiplied by the estimated shares. It gave +0.0017 to +0.0019 on the public leaderboard.
4. Predictions are mapped to the `id` of the corrected `test.csv` through the image file name.

**Input.** The member probability files (`CPE342_T4_PROBS`, see `README.md`). **Output.** `../outputs/task4_pred_v57a.npy` (final file V57a), `task4_pred_v57a_noEM.npy` (same members without step 3) and, if the member exists, `task4_pred_v53c.npy` (the second selected file).

In [1]:
import os, re, sys, time, warnings
from pathlib import Path
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")

# ---- configuration -------------------------------------------------------------------------------
# DATA = folder of the Kaggle download that holds 01_binary_classification/ ... 05_anomaly_detection/ and sample_submission.csv
DATA = Path(os.environ.get("CPE342_DATA", "")).expanduser()
assert (DATA / "01_binary_classification").exists(), "Set the environment variable CPE342_DATA to the Kaggle download folder (see ../README.md)"
OUT = Path(os.environ.get("CPE342_OUT", "../outputs")); OUT.mkdir(parents=True, exist_ok=True)   # shared by all notebooks
SEED = 342                                   # every random split / model seed in the project
NJOBS = int(os.environ.get("NJOBS", 16))     # LightGBM threads
PROB_DIR = Path(os.environ.get("CPE342_T4_PROBS", "../outputs/task4_probs"))                 # folder with t4_<tag>_test_prob.npy
T4_CSV_DIR = Path(os.environ.get("CPE342_T4_CSV_DIR", DATA / "04_image_classification"))      # folder with the corrected test.csv (id, file_name)
PROB_ORDER = os.environ.get("CPE342_T4_PROB_ORDER", "v1")      # row order of the probability files: "v1" = original test.csv (our saved members), "ans" = corrected test.csv (files written by train_backbone.py)
V1_TEST_CSV = Path(os.environ.get("CPE342_T4_V1_TEST_CSV", "../../../../cpe342-kaggle-main/04_image_classification/test.csv"))   # original test.csv (id = image name), in the repository
print("member probabilities in:", PROB_DIR.resolve(), "| order:", PROB_ORDER)

member probabilities in: C:\Users\Wirachat\Desktop\CPE342-KAGGLE\runs\wirachat\round-3 | order: v1


## 1. Members and weights

In [2]:
OTHERS = {"cnxtb22k_fresh": 4, "cnxtb22k_fresh_s5": 4, "cnxtb22k_hires": 4, "dinov2b_fresh": 4, "dinov2b_fresh_s7": 4, "dinov2b_fresh_s3": 4, "cnxts_fresh": 2, "cnxt_fresh": 2}
PLAIN_L = {t: 7 for t in ["dinov2l_fresh"] + [f"dinov2l_fresh_s{s}" for s in (33, 44, 55, 88, 99, 111, 122, 133, 144)]}      # baseline recipe, ten seeds
NEW_L = {f"dinov2l_new_s{s}": 33 for s in (201, 202, 203, 204, 205, 206)}                                                      # new recipe (llrd + EMA + CutMix/MixUp), six seeds
NEW_B = {"dinov2b_new_s301": 20}
FINAL = {**OTHERS, **PLAIN_L, **NEW_L, **NEW_B}                                                                                # file V57a
ALT = {**OTHERS, **PLAIN_L, **{f"dinov2l_new_s{s}": 49 for s in (201, 202, 203, 204)}, "exp_b_sink": 20}                         # file V53c (four new-recipe L + a DINOv2-B trained on 80% of the images)
tot = sum(FINAL.values()); print(f"V57a: {len(FINAL)} members, total weight {tot}; new-recipe share {(sum(NEW_L.values()) + sum(NEW_B.values())) / tot:.0%}")

def load_members(spec):
    P = sum(w * np.load(PROB_DIR / f"t4_{t}_test_prob.npy") for t, w in spec.items()) / sum(spec.values())
    return P
def em_prior(P, iters=200):
    """Saerens EM: estimate the class priors of the test set from the predicted probabilities (training loss was class-balanced, so start from uniform)."""
    pi = np.ones(P.shape[1]) / P.shape[1]
    for _ in range(iters):
        Q = P * (pi / (1 / P.shape[1])); Q /= Q.sum(1, keepdims=True); pi = Q.mean(0)
    Q = P * (pi / (1 / P.shape[1])); return Q / Q.sum(1, keepdims=True), pi
share = lambda a: np.round(np.bincount(a, minlength=5) / len(a), 3).tolist()

V57a: 25 members, total weight 316; new-recipe share 69%


## 2. Ensemble, class-prior correction and mapping to the test ids

In [3]:
P = load_members(FINAL)
P_em, pi = em_prior(P)
print("ensemble class shares (argmax):", share(P.argmax(1)), "| EM estimate of the test priors:", np.round(pi, 3).tolist())
print("after the class-prior correction:", share(P_em.argmax(1)), "| rows changed:", int((P.argmax(1) != P_em.argmax(1)).sum()))

test = pd.read_csv(T4_CSV_DIR / "test.csv"); stems = test.file_name.str.replace(".jpg", "", regex=False).tolist()
order = pd.read_csv(V1_TEST_CSV).id.astype(str).tolist() if PROB_ORDER == "v1" else stems     # image names in the row order of the probability files
assert len(order) == len(P)
def to_ans(pred):                                                                              # map the predictions by image name into the order of the corrected test.csv
    by_img = dict(zip(order, pred)); return np.array([by_img[s] for s in stems], dtype=int)
np.save(OUT / "task4_pred_v57a.npy", to_ans(P_em.argmax(1))); np.save(OUT / "task4_pred_v57a_noEM.npy", to_ans(P.argmax(1)))
print("saved task4_pred_v57a.npy (class shares", share(to_ans(P_em.argmax(1))), ")")

ensemble class shares (argmax): [0.146, 0.273, 0.152, 0.24, 0.188] | EM estimate of the test priors: [0.095, 0.301, 0.135, 0.26, 0.21]
after the class-prior correction: [0.135, 0.289, 0.139, 0.249, 0.187] | rows changed: 788


saved task4_pred_v57a.npy (class shares [0.135, 0.289, 0.139, 0.249, 0.187] )


## 3. The second selected file (V53c)

In [4]:
if all((PROB_DIR / f"t4_{t}_test_prob.npy").exists() for t in ALT):
    P2_em, _ = em_prior(load_members(ALT)); np.save(OUT / "task4_pred_v53c.npy", to_ans(P2_em.argmax(1)))
    print("V53c: saved task4_pred_v53c.npy | rows that differ from V57a:", int((to_ans(P2_em.argmax(1)) != np.load(OUT / "task4_pred_v57a.npy")).sum()))
else:
    print("V53c members not found; skipped")

V53c: saved task4_pred_v53c.npy | rows that differ from V57a:

 258
